In [9]:
# Basis
from pathlib import Path
import numpy as np
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import contextily as cx
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [10]:
# and from hydrolib-core
from hydrolib.core.dimr.models import DIMR, FMComponent
from hydrolib.core.dflowfm.inifield.models import IniFieldModel, DiskOnlyFileModel
from hydrolib.core.dflowfm.onedfield.models import OneDFieldModel
from hydrolib.core.dflowfm.structure.models import StructureModel
from hydrolib.core.dflowfm.crosssection.models import CrossLocModel, CrossDefModel
from hydrolib.core.dflowfm.ext.models import ExtModel
from hydrolib.core.dflowfm.mdu.models import FMModel
from hydrolib.core.dflowfm.friction.models import FrictionModel
from hydrolib.core.dflowfm.obs.models import ObservationPointModel
from hydrolib.core.dflowfm.storagenode.models import StorageNodeModel

In [11]:
from hydrolib.dhydamo.core.hydamo import HyDAMO
from hydrolib.dhydamo.converters.df2hydrolibmodel import Df2HydrolibModel
from hydrolib.dhydamo.geometry import mesh
from hydrolib.dhydamo.core.drr import DRRModel
from hydrolib.dhydamo.core.drtc import DRTCModel
from hydrolib.dhydamo.io.dimrwriter import DIMRWriter
from hydrolib.dhydamo.io.drrwriter import DRRWriter
from hydrolib.dhydamo.geometry.viz import plot_network
from meshkernel.py_structures import DeleteMeshOption

Define in- and output paths

In [12]:
# path to the package containing the dummy-data
dir_data = Path("..\\..\\WRIJ_RR_Unpaved_methode_02_tussenresultaat\\")
dir_input_rr_ernst = Path(dir_data, "RR_input_rr_ernst\\")
dir_input_meteo = Path(dir_data, "RR_input_meteo\\")

In [13]:
# overwrite output-path to write the models
output_path = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\test_oude_ijssel\\")
if not output_path.exists():
    output_path.mkdir(parents=True)

## Rainfall runoff model

RR has not changed yet compared to delft3dfmpy. Initialize a model:

In [14]:
drrmodel = DRRModel()

In [ ]:
df_unpaved_zomer = pd.read_csv(dir_input_rr_ernst / f"df_unpaved_zomer.csv")
df_unpaved_winter = pd.read_csv(dir_input_rr_ernst / f"df_unpaved_winter.csv")
df_ernst_zomer = pd.read_csv(dir_input_rr_ernst / f"df_ernst_zomer.csv")
df_ernst_winter = pd.read_csv(dir_input_rr_ernst / f"df_ernst_winter.csv")

### Unpaved nodes

Unpaved_from_input has now an optional argument containing the greenhouse areas: if a catchment intersects them its area (the most ocurring class) is corrected for the greenhouse area. 

In [16]:
for unpaved_dict in df_unpaved_zomer.set_index("code").to_dict("records"):
    drrmodel.unpaved.add_unpaved(**unpaved_dict)

for ernst_dict in df_ernst_zomer.set_index("code").to_dict("records"):
    drrmodel.unpaved.add_ernst_def(**ernst_dict)

### Open water

As opposed to Sobek, in D-Hydro open water is merely an interface for precpitation and evaporation. No management and water levels are included.

In [17]:
# drrmodel.openwater.io.openwater_from_input(
#     hydamo.catchments, lu_file, meteo_areas, zonalstats_alltouched=True
# )

### Read HyDAMO DAMO2.2 data

Explore the geopackage

In [18]:
# initialize a hydamo object
hydamo = HyDAMO(extent_file=dir_data / "OudeIJsselOutline.gpkg")

Load branches and profiles.

In the funtions below, the function 'snap_to_branch_and_drop' compares each object with a geometry to the branches. If the object is outside the specified maximum distance to any branch, the object and all objects related to it are dropped (if 'drop_related' is True).

Moreover there are multiple options to snap:
- overal: for points, based on minimum distance to the branch;
- centroid: for lines and polygons, based on the mininimum distance of the objets' centroid to the branch;
- intersecting: for lines, takes the first branch the object is intersecting (for lines);
- ends: for lines, based on the cumulative distance of the lines' ends to the branch.

In [19]:
path_watergangen = Path(dir_data, "watergang.gpkg")
hydamo.branches.read_gpkg_layer(path_watergangen, layer_name="watergang", index_col="code")

Catchments and laterals

In [ ]:
# read catchments
path_afwateringseenheden = Path(dir_data, "afwateringseenheden.gpkg")
hydamo.catchments.read_gpkg_layer(path_afwateringseenheden, layer_name="afwateringseenheden", index_col="code", check_geotype=False)

,code,geometry,globalid,lateraleknoopid,boundary_node
code,,,,,
AE54910021,AE54910021,"MULTIPOLYGON Z (((220466.705 440575.018 0, 220...",{99CFBB12-3EC8-4385-BAEA-4FBE9F6C838A},lateral_AE54910021,NaN
AE54150004,AE54150004,"MULTIPOLYGON Z (((239666.903 436131.017 0, 239...",{36DF26D9-2407-4B8B-AF71-CBE47F47C408},lateral_AE54150004,NaN
AE54680110,AE54680110,"MULTIPOLYGON Z (((229110.747 444029.643 0, 229...",{AFEA87FC-7D6B-426A-AE3D-A06CC5A5C638},lateral_AE54680110,NaN
AE54640162,AE54640162,"MULTIPOLYGON Z (((246287.875 440161.345 0, 246...",{A6104CE1-E5CC-49CE-8582-B0EC17005A7B},lateral_AE54640162,NaN
AE54770013,AE54770013,"MULTIPOLYGON Z (((229517.681 439267.634 0, 229...",{F948DB61-9497-4BE3-A401-7973B8FF246D},lateral_AE54770013,NaN
...,...,...,...,...,...
AE54182720_DE00,AE54182720_DE00,"MULTIPOLYGON Z (((245328.876 429904.005 0, 245...",{16D81EB7-009B-4934-95F2-AAACCC5E75D5},lateral_AE54182720_DE00,NaN
AE54330033,AE54330033,"MULTIPOLYGON Z (((242162.806 439233.97 0, 2422...",{8105E3CC-C3F9-4BB5-AB16-299EBEEAA55B},lateral_AE54330033,NaN
AE54910079,AE54910079,"MULTIPOLYGON Z (((218549.21 441939.236 0, 2185...",{BD6B7168-4F62-4E49-BBA9-E4519B59D27C},lateral_AE54910079,NaN


In [37]:
laterals_gdf = gpd.GeoDataFrame(df_unpaved_zomer, geometry=gpd.points_from_xy(df_unpaved_zomer.px-10, df_unpaved_zomer.py-10), crs=28992)
laterals_gdf["rr_node"] = laterals_gdf["code"]
laterals_gdf["code"] = laterals_gdf["boundary_node"]
laterals_gdf["globalid"] = laterals_gdf["code"]
laterals_gdf = laterals_gdf[["code", "rr_node", "globalid", "geometry"]].set_index("code")
laterals_gdf.to_file(dir_data / "laterale_knoop.gpkg", layer="laterale_knoop", driver="GPKG")

In [ ]:
# read laterals
hydamo.laterals.read_gpkg_layer(dir_data / "laterale_knoop.gpkg", layer_name="laterale_knoop")
hydamo.laterals.snap_to_branch(hydamo.branches, snap_method="overal", maxdist=5000)
hydamo.catchments['boundary_node'] = hydamo.catchments['lateraleknoopid'].copy()

In [59]:
meteo_areas = hydamo.catchments

### RR boundaries

They are different for the (paved) case with and without overflows. Overflows and greenhouse laterals are optional, but should be provided if they have been used above. 

In [60]:
drrmodel.external_forcings.io.boundary_from_input(
    hydamo.laterals, 
    hydamo.catchments, 
    drrmodel, 
)

87 catchments removed because of an area of 0 m2.


IndexError: single positional indexer is out-of-bounds

### External forcings

Three types of external forcing need to be provided:<br>
- Seepage/drainage
- Precipitation
- Evaporation

All are assumed to be spatially variable and thus need to pe provided as rasters per time step. Only the locations of the folders containing the rasters need to be provided; the time step is then derived from the file names.

Precipitation and evaporation are assumed to be in mm/d. As for evaporation only one meteostation is used, the meteo_areas are dissolved. For seepage, as the use of Metaswap-rasters is allowed, the unit is assumed to m3/grid cell/timestep.

Rastertypes can be any type that is recognized by rasterio (in any case Geotiff and ArcASCII rasters). If the file extension is 'IDF', as is the case in Modflow output, the raster is read using the 'imod'-package.

IMPORTANT: time steps are extracted from the file names. Therefore, the names should cohere to some conditions:
The filename should consist of at least two parts, separated by underscores. The second part needs to contain time information, which should be formatted as YYYYMMDDHHMMSS (SS may be omitted). Or, for daily data YYYYMMDD.

For example: 'precip_20200605151500.tif'

Extracting meteo-data from rasters can be time consuming. If precip_file and evap_file are specified, meteo-files are copied from an existing location.

In [ ]:
# seepage_folder = str(dir_input_meteo / "rasters" / "seepage")
precip_file = str(dir_input_meteo / "METEO_NEERSLAG.BUI")
evap_file = str(dir_input_meteo / "METEO_VERDAMPING.EVP")
# drrmodel.external_forcings.io.seepage_from_input(hydamo.catchments, seepage_folder)
drrmodel.external_forcings.io.precip_from_input(meteo_areas, precip_folder=None, precip_file=precip_file)
drrmodel.external_forcings.io.evap_from_input(meteo_areas, evap_folder=None, evap_file=evap_file)

Add the main parameters:

In [ ]:
drrmodel.d3b_parameters["Timestepsize"] = 300
drrmodel.d3b_parameters["StartTime"] = "'2016/06/01;00:00:00'"  # should be equal to refdate for D-HYDRO
drrmodel.d3b_parameters["EndTime"] = "'2016/06/03;00:00:00'"
drrmodel.d3b_parameters["RestartIn"] = 0
drrmodel.d3b_parameters["RestartOut"] = 0
drrmodel.d3b_parameters["RestartFileNamePrefix"] = "Test"
drrmodel.d3b_parameters["UnsaturatedZone"] = 1
drrmodel.d3b_parameters["UnpavedPercolationLikeSobek213"] = -1
drrmodel.d3b_parameters["VolumeCheckFactorToCF"] = 100000

Laterals are different for the case with and without RR. There can be three options:
1) laterals from the RR model (RR=True). There will be real-time coupling where RR and FM are calculated in parallel. Note that, again, the overflows are needed because there are extra boundaries. If there are no overflows, it does not have to be provided.
2) timeseries: lateral_discharges can be a dataframe with the code of the lateral as column headers and timesteps as index
3) constant: lateral_discharges can be a pandas Series with the code of the lateral as the index. This is the case in the example when RR=False.

In [ ]:
hydamo.external_forcings.convert.laterals(
    hydamo.laterals,
    # overflows=hydamo.overflows,
    #greenhouse_laterals=hydamo.greenhouse_laterals,
    # lateral_discharges=None,
    rr_boundaries=drrmodel.external_forcings.boundary_nodes
)

### Plot the RR model

In [ ]:
def node_geometry(dict):
    # Function to put the node geometries in geodataframes
    from shapely.geometry import Point, LineString

    geoms = []
    links = []
    for i in dict.items():
        if "ar" in i[1]:
            if np.sum([float(s) for s in i[1]["ar"].split(" ")]) > 0:
                geoms.append(Point((float(i[1]["px"]), float(i[1]["py"]))))
                links.append(
                    LineString(
                        (
                            Point(float(i[1]["px"]), float(i[1]["py"])),
                            Point(
                                float(drrmodel.external_forcings.boundary_nodes[i[1]["boundary_node"]]["px"]),
                                float(drrmodel.external_forcings.boundary_nodes[i[1]["boundary_node"]]["py"]),
                            ),
                        )
                    )
                )
        else:
            geoms.append(Point((float(i[1]["px"]), float(i[1]["py"]))))
    return ((gpd.GeoDataFrame(geoms, columns=["geometry"])), gpd.GeoDataFrame(links, columns=["geometry"]))

In [ ]:
plt.rcParams['axes.edgecolor'] = 'w'
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))

ax.xaxis.set_visible(False)
ax.yaxis.set_visible(False)
xmin,ymin,xmax,ymax=hydamo.clipgeo.bounds
ax.set_xlim(round(xmin), round(xmax))
ax.set_ylim(round(ymin), round(ymax))

hydamo.catchments.geometry.plot(ax=ax, label="Catchments", edgecolor="black", facecolor="pink", alpha=0.5)
hydamo.branches.geometry.plot(ax=ax, label="Channel")
node_geometry(drrmodel.unpaved.unp_nodes)[0].plot(
    ax=ax, markersize=30, marker="s", color="green", label="Unpaved"
)
node_geometry(drrmodel.unpaved.unp_nodes)[1].plot(ax=ax, color="black", linewidth=0.5)
node_geometry(drrmodel.paved.pav_nodes)[0].plot(ax=ax, markersize=20, marker="s", color="red", label="Paved")
node_geometry(drrmodel.paved.pav_nodes)[1].plot(ax=ax, color="black", linewidth=0.5)
node_geometry(drrmodel.greenhouse.gh_nodes)[0].plot(ax=ax, markersize=15, color="yellow", label="Greenhouse")
node_geometry(drrmodel.greenhouse.gh_nodes)[1].plot(ax=ax, color="black", linewidth=0.5)
node_geometry(drrmodel.openwater.ow_nodes)[0].plot(ax=ax, markersize=10, color="blue", label="Openwater")
node_geometry(drrmodel.openwater.ow_nodes)[1].plot(ax=ax, color="black", linewidth=0.5, label="RR-link")
node_geometry(drrmodel.external_forcings.boundary_nodes)[0].plot(
    ax=ax, markersize=15, color="purple", label="RR Boundary"
)

# manually add handles for polygon plot
handles, labels = ax.get_legend_handles_labels()
poly = mpatches.Patch(facecolor="pink", edgecolor="black", alpha=0.5)    
ax.legend(handles=handles.append(poly), labels=labels.append("Catchments"))
cx.add_basemap(ax, crs=28992, source=cx.providers.OpenStreetMap.Mapnik)
fig.tight_layout()

## Writing the model

In D-Hydro the 1D timestep (dt user) should be at least equal to than the smallest timestep of RR and RTC, otherwise water balance problems may occur.
The following code sets 'dtuser' equal to the smallest time step.

In [ ]:
# check the timesteps:
timesteps = []
timesteps.append(drrmodel.d3b_parameters['Timestepsize'])

In [ ]:
rr_writer = DRRWriter(drrmodel, output_dir=output_path, name="test", wwtp=(199000.0, 396000.0))
rr_writer.write_all()

A run.bat that will run DIMR is written by the following command. Adjust this with your local D-Hydro Suite version.

In [ ]:
# dimr = DIMRWriter(output_path=output_path, dimr_path=str(r"C:\Program Files\Deltares\D-HYDRO Suite 2024.03 1D2D\plugins\DeltaShell.Dimr\kernels\x64\bin\run_dimr.bat"))

In [ ]:
# if not RR:
#     drrmodel = None
# if not RTC:
#     drtcmodel = None

In [ ]:
# dimr.write_dimrconfig(fm, rr_model=drrmodel, rtc_model=drtcmodel)

Add projection information (Rijksdriehoeksstelsel) to the net.nc-file.

In [ ]:
# dimr.add_crs()

In [ ]:
# dimr.write_runbat()

In [ ]:
# print("Done!")